# Enzyme Software — LNN Training on Google Colab

> **Run all cells top-to-bottom.** Mount Drive (Cell 3) before training so checkpoints survive a disconnect.

---

## GPU Cost Guide (Colab Pro)

| GPU | VRAM | Units/hr | Est. total time | Est. total units |
|-----|------|----------|-----------------|------------------|
| **T4** | 16 GB | ~2 | ~2–3 hrs | **~4–6** ✅ cheapest |
| **L4** | 24 GB | ~2.5 | ~1.5 hrs | **~3.7** ⭐ best value |
| **A100 40 GB** | 40 GB | ~6 | ~1 hr | **~6** |
| **A100 80 GB** | 80 GB | ~9 | ~50 min | **~7.5** |
| **H100** | 80 GB | ~13 | ~30 min | **~6.5** |

**Recommendation:** Use **T4** (fewest units used) or **L4** (best speed-per-unit).  
This model is ~2M parameters trained on small drug molecules — T4 is more than sufficient.

---

### What this trains (7-step pipeline)
1. **hybrid_lnn** — core LTC graph network (CYP + SoM, cross-atom attention, BDE prior)
2. **hybrid_full_xtb** — fine-tune with full xTB quantum features
3. **micropattern_xtb** — fine-tune with micropattern + xTB features
4. **extract predictions** — run all 3 models on dataset, stack outputs
5. **CAHML OOF** — 5-fold out-of-fold CAHML predictions (no data leakage)
6. **multihead meta-learner** — train on 4-expert OOF stack
7. **CAHML final** — full-data CAHML for inference


In [ ]:
# ── Cell 1: Check GPU and auto-select batch size ──────────────────────────────
import subprocess

try:
    gpu_info = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True
    ).stdout.strip()
    print('GPU:', gpu_info)
except Exception:
    gpu_info = ''
    print('No GPU detected!')
    print('Go to: Runtime > Change runtime type > Hardware accelerator > GPU')

if any(x in gpu_info for x in ['A100', 'H100']):
    BATCH = 32
    print('Tier: high-end (A100/H100) -> BATCH=64')
elif any(x in gpu_info for x in ['L4', 'V100', 'P100']):
    BATCH = 32
    print('Tier: mid-range (L4/V100) -> BATCH=32')
else:
    BATCH = 32
    print('Tier: T4 or unknown -> BATCH=32')

print()
print('You can override BATCH in the Config cell (Cell 5) if needed.')


In [ ]:
# ── Cell 2: Clone repo and install dependencies ───────────────────────────────
import os, subprocess, importlib

REPO = 'https://github.com/Nikku03/enzyme_Software.git'
REPO_DIR = '/content/enzyme_Software'

if not os.path.exists(REPO_DIR):
    print('Cloning repo...')
    subprocess.run(f'git clone {REPO} {REPO_DIR}', shell=True, check=True)
    print('Cloned.')
else:
    print('Repo exists, pulling latest...')
    subprocess.run(f'git -C {REPO_DIR} pull', shell=True)

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

print()
print('Installing rdkit (takes ~1 min)...')
subprocess.run('pip install rdkit -q', shell=True, check=True)

print('Installing enzyme_software package...')
subprocess.run('pip install -e . -q', shell=True, check=True)

print()
print('Package versions:')
for name, mod in [('torch', 'torch'), ('rdkit', 'rdkit'), ('numpy', 'numpy')]:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, '__version__', 'ok')
        print(f'  {name}: {ver}')
    except ImportError:
        print(f'  {name}: NOT FOUND')


In [ ]:
# ── Cell 3: Mount Google Drive (checkpoints survive disconnect) ────────────────
# IMPORTANT: Run this before training. If you skip it, checkpoints are lost on disconnect.
import os, shutil

try:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_SAVE = '/content/drive/MyDrive/enzyme_lnn_training_2158'
    os.makedirs(f'{DRIVE_SAVE}/checkpoints', exist_ok=True)
    os.makedirs(f'{DRIVE_SAVE}/artifacts', exist_ok=True)

    os.chdir('/content/enzyme_Software')

    # Symlink local checkpoints/ and artifacts/ -> Drive
    # All training scripts use relative paths, so they write directly to Drive.
    for d in ['checkpoints', 'artifacts']:
        drive_path = f'{DRIVE_SAVE}/{d}'
        if os.path.islink(d):
            os.unlink(d)  # re-link (reconnect case)
        elif os.path.isdir(d):
            # Move any existing local data to Drive first
            shutil.copytree(d, drive_path, dirs_exist_ok=True)
            shutil.rmtree(d)
        os.symlink(drive_path, d)
        print(f'  {d}/ -> {drive_path}')

    print()
    print('Drive mounted. Checkpoints will survive Colab disconnects.')
    print(f'Find your files at: {DRIVE_SAVE}')

except Exception as e:
    print(f'Drive mount skipped: {e}')
    print()
    print('WARNING: checkpoints are LOCAL only — lost if Colab disconnects!')
    print('Tip: re-run this cell and click Allow when Drive asks for permission.')


In [ ]:
# ── Cell 4: Install XTB (optional — needed for Steps 2 & 3) ──────────────────
# WITHOUT XTB: Steps 2 & 3 will zero-pad quantum chemistry features.
#              Training still works and converges — you just miss the QC signal.
# WITH XTB:    Richer features for hybrid_full_xtb and micropattern_xtb models.
import subprocess, shutil, os

if shutil.which('xtb'):
    r = subprocess.run(['xtb', '--version'], capture_output=True, text=True)
    first_line = r.stdout.strip().split('\n')[0] if r.stdout.strip() else 'unknown version'
    print(f'XTB already installed: {first_line}')
else:
    print('XTB not found. Attempting install via precompiled binaries (~1 min)...')
    install_cmd = (
        'wget -q https://github.com/grimme-lab/xtb/releases/download/v6.7.1/xtb-6.7.1-linux-x86_64.tar.xz && '
        'tar -xf xtb-6.7.1-linux-x86_64.tar.xz -C /usr/local --strip-components=1 && '
        'rm xtb-6.7.1-linux-x86_64.tar.xz'
    )
    r1 = subprocess.run(install_cmd, shell=True, capture_output=True, text=True)

    if r1.returncode == 0 and shutil.which('xtb'):
        r = subprocess.run(['xtb', '--version'], capture_output=True, text=True)
        first_line = r.stdout.strip().split('\n')[0] if r.stdout.strip() else 'unknown version'
        print(f'XTB installed successfully: {first_line}')
    else:
        print('XTB install failed.')
        if r1.stderr:
            print(r1.stderr)
        print('  -> Steps 2 & 3 will use zero-padded features. Training still valid.')

In [ ]:
# ── Cell 5: Training configuration ───────────────────────────────────────────
# Edit values here. All training cells below read from these variables.
import os, subprocess, time

# ── Edit these ────────────────────────────────────────────────────────────────
DATASET = 'data/merged_all_sources.json'   # 2158-compound merged dataset (5 sources)
EPOCHS  = 75    # 50=fast (~1.5 hrs on T4), 75=good quality, 100=best (2.5+ hrs)
SEED    = 42
LR      = '2e-4'
WD      = '1e-4'
# BATCH auto-set by GPU check (Cell 1) — already set to 32 for T4.
# Override here if you want a different value:
# BATCH = 16
# ─────────────────────────────────────────────────────────────────────────────

os.chdir('/content/enzyme_Software')
os.environ['PYTHONPATH'] = os.getcwd() + '/src'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# Create all required directories
for d in [
    'logs',
    'cache/manual_engine_full',
    'cache/full_xtb',
    'cache/micropattern_xtb',
    'cache/meta_learner',
    'cache/meta_learner_multihead',
    'cache/cahml',
    'cache/cahml_oof_folds',
    'checkpoints/hybrid_full_xtb',
    'checkpoints/micropattern_xtb',
    'checkpoints/meta_learner_multihead',
    'checkpoints/cahml',
    'artifacts/hybrid_full_xtb',
    'artifacts/micropattern_xtb',
    'artifacts/meta_learner_multihead',
    'artifacts/cahml',
]:
    os.makedirs(d, exist_ok=True)


def run_step(name, cmd):
    """Run a training step with live output printed to the cell."""
    sep = '-' * 62
    print()
    print(sep)
    print(f'  {name}')
    print(sep)
    t0 = time.time()
    env = {**os.environ, 'PYTHONPATH': os.getcwd() + '/src', 'KMP_DUPLICATE_LIB_OK': 'TRUE'}
    proc = subprocess.Popen(
        cmd, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, env=env, bufsize=1
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    elapsed = (time.time() - t0) / 60
    if proc.returncode == 0:
        print(f'[DONE] {elapsed:.1f} min')
    else:
        print(f'[FAILED — exit code {proc.returncode}] {elapsed:.1f} min')
        raise RuntimeError(f'{name} failed with exit code {proc.returncode}')


print('Config ready:')
print(f'  DATASET  : {DATASET}')
print(f'  EPOCHS   : {EPOCHS}')
print(f'  BATCH    : {BATCH}')
print(f'  LR / WD  : {LR} / {WD}')
print(f'  SEED     : {SEED}')
print(f'  checkpoints/ : {os.path.realpath("checkpoints")}')


In [ ]:
# ── Step 1 / 7: hybrid_lnn — core LTC model ──────────────────────────────────
# This is the main model. Takes the longest (40-60% of total training time).
# --auto-resume-latest: automatically continues from last checkpoint if interrupted.
run_step('Step 1/7 — hybrid_lnn', ' '.join([
    'python scripts/train_hybrid_lnn.py',
    f'--dataset {DATASET}',
    '--train-ratio 0.70',
    '--val-ratio 0.20',
    f'--epochs {EPOCHS}',
    f'--early-stopping-patience {EPOCHS}',
    f'--learning-rate {LR}',
    f'--weight-decay {WD}',
    f'--batch-size {BATCH}',
    f'--seed {SEED}',
    '--output-dir checkpoints',
    '--manual-feature-cache-dir cache/manual_engine_full',
    '--auto-resume-latest',
]))


In [ ]:
# ── Step 2 / 7: hybrid_full_xtb — fine-tune with full xTB features ────────────
# Fine-tunes from hybrid_lnn_best.pt with richer quantum chemistry features.
# If XTB is not installed: zero-pads quantum features (training still valid).
run_step('Step 2/7 — hybrid_full_xtb', ' '.join([
    'python scripts/train_hybrid_full_xtb.py',
    f'--dataset {DATASET}',
    '--checkpoint checkpoints/hybrid_lnn_best.pt',
    '--xtb-cache-dir cache/full_xtb',
    '--train-ratio 0.70',
    '--val-ratio 0.15',
    f'--epochs {EPOCHS}',
    f'--early-stopping-patience {EPOCHS}',
    '--learning-rate 5e-5',
    f'--weight-decay {WD}',
    f'--batch-size {BATCH}',
    f'--seed {SEED}',
    '--output-dir checkpoints/hybrid_full_xtb',
    '--artifact-dir artifacts/hybrid_full_xtb',
    '--compute-xtb-if-missing',
]))


In [ ]:
# ── Step 3 / 7: micropattern_xtb — fine-tune with micropattern features ───────
# Specializes on site-labeled molecules only, adding micropattern features.
# If XTB is not installed: zero-pads quantum features (training still valid).
run_step('Step 3/7 — micropattern_xtb', ' '.join([
    'python scripts/train_micropattern_xtb.py',
    f'--dataset {DATASET}',
    '--checkpoint checkpoints/hybrid_lnn_best.pt',
    '--xtb-cache-dir cache/micropattern_xtb',
    '--train-ratio 0.70',
    '--val-ratio 0.15',
    f'--epochs {EPOCHS}',
    '--learning-rate 5e-5',
    f'--weight-decay {WD}',
    f'--batch-size {BATCH}',
    f'--seed {SEED}',
    '--site-labeled-only',
    '--compute-xtb-if-missing',
    '--output-dir checkpoints/micropattern_xtb',
    '--artifact-dir artifacts/micropattern_xtb',
]))


In [ ]:
# ── Step 4 / 7: Extract base predictions (stack all 3 models) ─────────────────
# Runs hybrid_lnn, hybrid_full_xtb, and micropattern_xtb on the full dataset
# and saves per-molecule atom-score predictions for the meta-learner.
run_step('Step 4/7 — extract base predictions', ' '.join([
    'python scripts/extract_base_predictions.py',
    f'--dataset {DATASET}',
    '--output cache/meta_learner/base_predictions.pt',
    '--model-names hybrid_lnn hybrid_full_xtb micropattern_xtb',
    f'--seed {SEED}',
]))


In [ ]:
# ── Step 5 / 7: CAHML OOF — 5-fold leak-free stacking ────────────────────────
# Trains CAHML in 5 cross-validation folds to generate out-of-fold predictions.
# This prevents data leakage when the meta-learner is trained on CAHML output.
run_step('Step 5/7 — CAHML OOF predictions (5-fold)', ' '.join([
    'python scripts/generate_oof_cahml_predictions.py',
    '--predictions cache/meta_learner/base_predictions.pt',
    f'--dataset {DATASET}',
    '--output cache/meta_learner/base_predictions_cahml_oof.pt',
    '--n-folds 5',
    f'--epochs {EPOCHS}',
    '--patience 15',
    '--learning-rate 1e-3',
    f'--weight-decay {WD}',
    '--hidden-dim 64',
    '--mirank-weight 1.0',
    '--bce-weight 0.3',
    '--listmle-weight 0.5',
    '--focal-weight 0.2',
    f'--seed {SEED}',
    '--work-dir cache/cahml_oof_folds',
]))


In [ ]:
# ── Step 6 / 7: Multihead meta-learner (4-expert ensemble) ────────────────────
# Trains a small meta-learner on the 4-expert OOF stack
# (hybrid_lnn + hybrid_full_xtb + micropattern_xtb + CAHML).
run_step('Step 6/7 — multihead meta-learner', ' '.join([
    'python scripts/train_multihead_meta_learner.py',
    '--predictions cache/meta_learner/base_predictions_cahml_oof.pt',
    f'--dataset {DATASET}',
    '--train-ratio 0.70',
    '--val-ratio 0.20',
    f'--epochs {EPOCHS}',
    f'--patience {EPOCHS}',
    '--learning-rate 1e-3',
    f'--weight-decay {WD}',
    '--hidden-dim 64',
    '--mirank-weight 1.0',
    '--bce-weight 0.3',
    f'--seed {SEED}',
    '--output-dir checkpoints/meta_learner_multihead',
    '--artifact-dir artifacts/meta_learner_multihead',
    '--cache-dir cache/meta_learner_multihead',
]))


In [ ]:
# ── Step 7 / 7: CAHML final — full-data training for inference ────────────────
# Trains CAHML on the full dataset (not OOF) for use during inference.
run_step('Step 7/7 — CAHML final', ' '.join([
    'python scripts/train_cahml.py',
    '--predictions cache/meta_learner/base_predictions.pt',
    f'--dataset {DATASET}',
    '--train-ratio 0.70',
    '--val-ratio 0.15',
    f'--epochs {EPOCHS}',
    f'--patience {EPOCHS}',
    '--learning-rate 1e-3',
    f'--weight-decay {WD}',
    '--hidden-dim 64',
    '--mirank-weight 1.0',
    '--bce-weight 0.3',
    '--listmle-weight 0.5',
    '--focal-weight 0.2',
    f'--seed {SEED}',
    '--output-dir checkpoints/cahml',
    '--artifact-dir artifacts/cahml',
    '--cache-dir cache/cahml',
]))


In [ ]:
# ── Training complete — show saved checkpoints ────────────────────────────────
import os, glob

print('='*62)
print('  ALL 7 STEPS COMPLETE')
print('='*62)
print()
print('Saved checkpoints:')

patterns = [
    'checkpoints/hybrid_lnn_best.pt',
    'checkpoints/hybrid_full_xtb/*best*.pt',
    'checkpoints/micropattern_xtb/*best*.pt',
    'checkpoints/meta_learner_multihead/*best*.pt',
    'checkpoints/cahml/*best*.pt',
]
found = 0
for pattern in patterns:
    for path in sorted(glob.glob(pattern)):
        size_mb = os.path.getsize(path) / 1e6
        real = os.path.realpath(path)
        print(f'  {path}  ({size_mb:.1f} MB)')
        if '/drive/' in real:
            print(f'    -> saved to Google Drive')
        found += 1

if found == 0:
    print('  (no checkpoints found — check for errors above)')

print()
print('To use on reconnect:')
print('  1. Re-run Cell 3 (Drive mount) to re-link checkpoints/')
print('  2. Re-run Cell 5 (Config) to restore run_step() and env vars')
print('  3. Jump to any step cell and run from there')


In [ ]:
# ── Cell 14: Compare new vs old — keep or discard new checkpoints ─────────────
# Run this after Cell 13 finishes. Shows old (1449-compound) vs new (2158-compound)
# training stats. You decide whether to promote the new checkpoints.
import json, os, glob

OLD_ARTIFACTS = '/content/drive/MyDrive/enzyme_lnn_training/artifacts'
NEW_ARTIFACTS = '/content/drive/MyDrive/enzyme_lnn_training_2158/artifacts'

MODEL_KEYS = {
    'hybrid_full_xtb': ('best_val_top1', 'test_metrics.site_top1_acc'),
    'cahml':           ('best_val_top1',),
    'meta_learner_multihead': ('best_val_top1',),
}

def read_final_report(artifact_dir, model):
    pattern = os.path.join(artifact_dir, model, '*.json')
    files = sorted(glob.glob(pattern))
    if not files:
        return None
    with open(files[-1]) as f:
        return json.load(f)

def get_val(d, dotkey):
    parts = dotkey.split('.')
    for p in parts:
        if d is None:
            return None
        d = d.get(p)
    return d

print(f"{'Model':<30} {'Metric':<35} {'OLD (1449)':>12} {'NEW (2158)':>12} {'Winner':>8}")
print('-' * 105)

promote = True
for model, keys in MODEL_KEYS.items():
    old_r = read_final_report(OLD_ARTIFACTS, model)
    new_r = read_final_report(NEW_ARTIFACTS, model)
    for k in keys:
        old_v = get_val(old_r, k) if old_r else None
        new_v = get_val(new_r, k) if new_r else None
        old_s = f'{old_v:.4f}' if isinstance(old_v, float) else 'N/A'
        new_s = f'{new_v:.4f}' if isinstance(new_v, float) else 'N/A'
        if isinstance(old_v, float) and isinstance(new_v, float):
            winner = '✅ NEW' if new_v >= old_v else '⚠️  OLD'
            if new_v < old_v:
                promote = False
        else:
            winner = '?'
        print(f'{model:<30} {k:<35} {old_s:>12} {new_s:>12} {winner:>8}')

print()
if promote:
    print('✅ New models are equal or better on all metrics.')
    print('   To use them: download enzyme_lnn_training_2158/checkpoints/ from Drive')
    print('   and replace your local checkpoints/ folder.')
else:
    print('⚠️  Old models are better on at least one metric.')
    print('   Your original enzyme_lnn_training/ on Drive is UNTOUCHED.')
    print('   To discard new: delete enzyme_lnn_training_2158/ from Drive.')
    print('   To keep new anyway: download enzyme_lnn_training_2158/checkpoints/')
